In [ ]:
import os
import pandas
import numpy as np
!pip install --upgrade numpy
import os

# Preparing Dataset for both Audio and Text
--------------------------------------------------------

In [ ]:
root_path = '/kaggle/input/iemocap-csv'

df = pandas.read_csv(os.path.join(root_path, 'iemocap.csv'))
sessions = [1, 2, 3, 4, 5]
df = df[df['session'].isin(sessions)]

# Remove unwanted emotions and empty values
unwanted_emotions = ['xxx', '', 'oth', 'dis', 'sur', 'fea', 'exc', 'fru']
df = df[~df['emotion'].isin(unwanted_emotions)]

# Calculate annotator difference
df['annotator_difference'] = df['n_annotators'] - df['agreement']

# Filter by annotator difference
df = df[df['annotator_difference'] <= 1]

# Replace 'exc' emotion with 'hap'
#df.loc[df['emotion'] == 'exc', 'emotion'] = 'hap'


emotions_count_before = df['emotion'].value_counts()
print("Emotions count before filtering:")
print(emotions_count_before)

# Group by emotion and select first 550 rows of each group
df = df.groupby('emotion').head(650)

# Count the occurrences of each emotion after filtering
emotions_count_after = df['emotion'].value_counts()
print("\nEmotions count after filtering:")
print(emotions_count_after)

# Display the first 5 rows
display(df)

In [ ]:
from sklearn.model_selection import train_test_split

df_train, df_valid = train_test_split(df, test_size=0.1, random_state=42)

print("Shape of df_train:", df_train.shape)
print("Shape of df_valid:", df_valid.shape)

In [ ]:
def split_and_convert_path(wav_path):
    parts = wav_path.split('/')
    main_path_parts = parts[:-3]  # Extract main path parts
    session_part = main_path_parts[1]  # Extract the session part (e.g., 'Session1')
    script_part = parts[-2]  # Extract the script part (e.g., 'Ses01F_script02_1')
    main_path = f'IEMOCAP_full_release/{session_part}/dialog/transcriptions/{script_part}.txt'  # Construct the main path
    file_name = parts[-1]  # Extract file name
    return main_path, file_name

df_train['main_path'], df_train['file_name'] = zip(*df_train['wav_path'].apply(split_and_convert_path))
df_valid['main_path'], df_valid['file_name'] = zip(*df_valid['wav_path'].apply(split_and_convert_path))

display(df_train[['main_path', 'file_name', 'emotion']])


In [ ]:
import re

# Assuming your DataFrame is called 'df'
def get_transcript(row):
    file_path = row['main_path']
    wav_file = row['file_name']
    
    file_path = os.path.join('/kaggle/input/iemocapfullrelease',file_path)
    
    with open(file_path, 'r') as f:
        transcript = ''
        for line in f:
            match = re.match(r'(.*?)\s\[(.*?)\]:\s(.*)', line)
            if match:
                speaker_id = match.group(1)
                if speaker_id in wav_file:
                    transcript += match.group(3) + ' '
    
    return transcript.strip()

df_train['text'] = df_train.apply(get_transcript, axis=1)
df_valid['text'] = df_valid.apply(get_transcript, axis=1)


In [ ]:
display(df_train[['wav_path', 'emotion', 'text']])

In [ ]:
from datasets import Dataset, DatasetDict, Features, Value, ClassLabel

# Define the class names
class_names = ['ang', 'hap', 'neu', 'sad']  # Update with your actual class names

# Define the features
features = Features({
    'emotion': ClassLabel(names=class_names),
    'text': Value('string')
})

# Preprocess the 'emotion' column in the dataframes
df_train['emotion'] = df_train['emotion'].apply(lambda x: class_names.index(x))
df_valid['emotion'] = df_valid['emotion'].apply(lambda x: class_names.index(x))

# Create individual datasets from the dataframes
train_dataset = Dataset.from_pandas(df_train, features=features)
valid_dataset = Dataset.from_pandas(df_valid, features=features)

# Create the DatasetDict
emotions = DatasetDict({
    'train': train_dataset,
    'validation': valid_dataset,
})

#  ---------------------------**Audio**---------------------------

In [ ]:
def get_labels(annot_file, file_name):

    f = open(annot_file, 'r').read()
    f = f.split('\n')
    f = f[2:]

    for data in f:

        if len(data) > 0:
            if data[0] == '[':
                data2 = data.split('\t')

                if data2[1] == file_name:
                    emo = data2[2]
                    vad = data2[3][1:-1].split(', ')
                    return emo, [float(x) for x in vad]

    raise ValueError('Label not found')

def get_mocap_rot(path):

    f = open(path, 'r').read()
    f = np.array(f.split('\n'))
    header = f[0].split(' ')
    xyz = f[1].split(' ')
    f = f[2:]

    data_list = []

    for data in f:
        data2 = data.split(' ')
        if(len(data2)<2):
            continue
        dic = {'frame': data2[0], 'time': data2[1],
               'markers': np.array(data2[2:]).astype(float)}
        data_list.append(dic)

    return header, xyz, data_list

def get_wav(path):
    x, sr = librosa.load(path, sr=16000)
    return x, sr

def get_ph_fa(path):
    f = open(path, 'r').read()
    f = np.array(f.split('\n'))
    header = f[0].split()
    f = f[1:-2]
    data_list = []

    for data in f:
        data2 = data.split()
        dic = {'SFrm':np.array(data2[0]).astype(int),
               'EFrm':np.array(data2[1]).astype(int),
               'SegAScr':np.array(data2[2]).astype(int),
               'Phone':data2[3]}
        data_list.append(dic)

    return header, data_list

def frame_to_s(fr):
    return (fr+2)*10/1000

In [ ]:
import librosa
root_path1 = '/kaggle/input/iemocapfullrelease/'

for index, row in df.iterrows():

    session = row['session']
    method = row['method']
    gender = row['gender']
    emotion = row['emotion']
    n_annot = row['emotion']
    agreement = row['agreement']
    wav_path = row['wav_path']

    _, file_name = os.path.split(wav_path)

    annot_file = os.path.join(root_path1, 'IEMOCAP_full_release',
                              'Session'+str(session),
                              'dialog', 'EmoEvaluation',
                              file_name[:-9] + '.txt')

    # get labels
    emo, vad = get_labels(annot_file, file_name[:-4])

    # get MOCAP data
    header_rot, xyz_rot, data_rot = get_mocap_rot(os.path.join(root_path1, row['MOCAP_rotated_path']))

    # get audio data
    x, sr = get_wav(os.path.join(root_path1, row['wav_path']))

    # get transcription data
    header_ph, data_ph = get_ph_fa(os.path.join(root_path1, row['FA_ph_path']))

    break


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import librosa
import os

def preprocess_data(df, root_path1, max_pad_length):
    features = []  # List to store extracted features
    labels = []    # List to store corresponding emotion labels

    for index, row in df.iterrows():
        session = row['session']
        wav_path = row['wav_path']
        emotion = row['emotion']

        # Extract filename from WAV path
        _, file_name = os.path.split(wav_path)

        # Load audio data
        x, sr = get_wav(os.path.join(root_path1, wav_path))

        # Extract features (e.g., MFCCs)
        mfccs = librosa.feature.mfcc(y=x, sr=sr, n_mfcc=13)

        # Flatten MFCCs (you may choose to use other features as well)
        flattened_mfccs = mfccs.flatten()

        # Append features and labels to the lists
        features.append(flattened_mfccs)
        labels.append(emotion)

    # Remove features without corresponding labels
    filtered_features = []
    filtered_labels = []
    for feature, label in zip(features, labels):
        if label is not None:
            filtered_features.append(feature)
            filtered_labels.append(label)

    # Pad or truncate features to the maximum length
    padded_filtered_features = []
    for feature in filtered_features:
        if len(feature) < max_pad_length:
            padded_feature = np.pad(feature, (0, max_pad_length - len(feature)))
        else:
            padded_feature = feature[:max_pad_length]
        padded_filtered_features.append(padded_feature)

    X = np.array(padded_filtered_features)
    y = np.array(filtered_labels)

    return X, y

In [ ]:
X_train_audio, y_train_audio = preprocess_data(df_train, root_path1, max_pad_length=3000)
X_valid_audio, y_valid_audio = preprocess_data(df_valid, root_path1, max_pad_length=3000)

In [ ]:
def plot_confusion_matrix(y_model, y_true, labels, cmap='Blues'):
    
    cm = confusion_matrix(y_true, y_model, normalize='true')
    fig, ax = plt.subplots(figsize=(7, 7))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    disp.plot(ax=ax, cmap=cmap, colorbar=False)  # Specify the colormap using cmap parameter
    plt.title("Confusion matrix")
    plt.show()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from sklearn.dummy import DummyClassifier

In [ ]:
xgb_clf = XGBClassifier(n_estimators=220, random_state=42)
xgb_clf.fit(X_train_audio, y_train_audio)
xgb_predictions = xgb_clf.predict(X_valid_audio)
xgb_accuracy = accuracy_score(y_valid_audio, xgb_predictions)
print("XGBoost Accuracy:", xgb_accuracy)
print("XGBoost Classification Report:")
print(classification_report(y_valid_audio, xgb_predictions))

In [ ]:
plot_confusion_matrix(xgb_predictions, y_valid_audio, class_names)

#  ---------------------------**TEXT**----------------------------

# DilstilBERT Tokenizer
-------------------------------------------------------------------------------------------

In [ ]:
from transformers import AutoTokenizer

model_ckpt = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

tokenizer

In [ ]:
emotions.reset_format()

In [ ]:
# Tokenisation function
def tokenise(batch):
    return tokenizer(batch["text"], padding=True, truncation=True)

emotions_encoded = emotions.map(tokenise, batched=True, batch_size=None)
print(emotions_encoded["train"].column_names)

# DistilBERT Model 
-------------------------------------------------------------------------------------------

In [ ]:
import warnings; warnings.filterwarnings('ignore')
from transformers import AutoModel
import torch

model_ckpt = "distilbert-base-uncased"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoModel.from_pretrained(model_ckpt).to(device)

Ectracting the last hidden layer ([CLS])

In [ ]:
def extract_hidden_states(batch):
    # Encode text
    encoded_text = tokenizer(batch["text"], padding=True, truncation=True, return_tensors="pt")

    # Place model inputs on the GPU
    inputs = {k: v.to(device) for k, v in encoded_text.items()}

    # Extract last hidden states
    with torch.no_grad():
        last_hidden_state = model(**inputs).last_hidden_state

    # Return vector for [CLS] token
    return {"hidden_state": last_hidden_state[:, 0].cpu().numpy()}

# Extract last hidden states (faster w/ GPU)
emotions_hidden = emotions_encoded.map(extract_hidden_states, batched=True)
print(emotions_hidden["train"].column_names)

In [ ]:
emotions_hidden['train']

# Classifier TEXT
-------------------------------------------------------------------------------------------

In [ ]:
X_train_text = np.array(emotions_hidden["train"]["hidden_state"])
X_valid_text = np.array(emotions_hidden["validation"]["hidden_state"])
y_train_text = np.array(emotions_hidden["train"]["emotion"])
y_valid_text = np.array(emotions_hidden["validation"]["emotion"])
print(f'Training Dataset: {X_train_text.shape}')
print(f'Validation Dataset {X_valid_text.shape}')

In [ ]:
dummy_clf = DummyClassifier(strategy="most_frequent")
dummy_clf.fit(X_train_text, y_train_text)
print(f'accuracy: {dummy_clf.score(X_valid_text, y_valid_text)}')

In [ ]:
# XGBoost Classifier
xgb_clf = XGBClassifier(n_estimators=250, random_state=42)
xgb_clf.fit(X_train_text, y_train_text)
xgb_predictions = xgb_clf.predict(X_valid_text)
xgb_accuracy = accuracy_score(y_valid_text, xgb_predictions)
print("XGBoost Accuracy:", xgb_accuracy)
print("XGBoost Classification Report:")
print(classification_report(y_valid_text, xgb_predictions))

In [ ]:
plot_confusion_matrix(xgb_predictions, y_valid_text, class_names)

# Testing the text model 

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

text = ' I am sure I can win this game'

inputs = tokenizer(text, return_tensors='pt')
inputs = {k:v.to(device) for k,v in inputs.items()}

with torch.no_grad():
    text_hidden_state = model(**inputs).last_hidden_state[:,0]
    
text_test = np.array(text_hidden_state.cpu())

rf_predictions = xgb_clf.predict(text_test)

class_names = ['ang', 'hap', 'neu', 'sad']  
class_names[rf_predictions[0]]


# Classifying Audio + Text
---------------------------------------------------------------------

In [ ]:
scaler = StandardScaler()
X_train_text_scaled = scaler.fit_transform(X_train_text)
X_valid_text_scaled = scaler.transform(X_valid_text)

In [ ]:
scaler = StandardScaler()
X_train_audio_scaled = scaler.fit_transform(X_train_audio)
X_valid_audio_scaled = scaler.transform(X_valid_audio)

In [ ]:
X_train_combined = np.concatenate((X_train_text_scaled, X_train_audio_scaled), axis=1)
X_valid_combined = np.concatenate((X_valid_text_scaled, X_valid_audio_scaled), axis=1)

In [ ]:
array_shape = X_train_combined.shape

# The shape will be a tuple (num_samples, num_features)
num_samples, num_features = array_shape

# Print the number of samples and features
print("Number of samples:", num_samples)
print("Number of features per sample:", num_features)

0.41 audio only -- 0.614 text only -- 0.41 combined - Linear Regression

In [ ]:
# XGBoost Classifier
xgb_clf = XGBClassifier(n_estimators=350, random_state=42)
xgb_clf.fit(X_train_combined, y_train_audio)
xgb_predictions = xgb_clf.predict(X_valid_combined)
xgb_accuracy = accuracy_score(y_valid_audio, xgb_predictions)
print("XGBoost Accuracy:", xgb_accuracy)
print("XGBoost Classification Report:")
print(classification_report(y_valid_text, xgb_predictions))

In [ ]:
plot_confusion_matrix(xgb_predictions, y_valid_audio, class_names)

Accuracy: 0.73 - combined //  0.55 - audio(MFCC) // 0.66 - text(BERT) - Gradient Boosting - hap.19


Accuracy: 0.69 - combined //  0.57 - audio(MFCC) //  0.65 - text(BERT) - RF- hap.4. does not improve
